<a href="https://colab.research.google.com/github/hariharan-vs/Deep-learning-lab/blob/main/EXP_10___24BAD030__Chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "streamlit", "transformers", "torch", "sentencepiece"
])

0

In [4]:

# EXPERIMENT 10: BUILDING A SIMPLE LLM CHATBOT
# Name: Hariharan V S | Roll No: 24BAD030

import os, sys, time, subprocess, urllib.request
from google.colab import output

app_code = r'''
import streamlit as st
import torch
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

st.set_page_config(page_title="LLM Chatbot", page_icon="🤖")

st.title("🤖 Simple LLM Chatbot")
st.caption("Experiment 10 | Hariharan V S | 24BAD030")

@st.cache_resource
def load_model():
    name = "google/flan-t5-small"
    tokenizer = AutoTokenizer.from_pretrained(name)
    model = AutoModelForSeq2SeqLM.from_pretrained(name)
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.to(device)
    model.eval()
    return tokenizer, model, device

def generate_response(message, history=None):
    if not message or not message.strip():
        return "Please enter a message."

    tokenizer, model, device = load_model()

    context = ""
    if history:
        for item in history[-4:]:
            context += f"{item['role']}: {item['content']}\n"

    prompt = (
        "Answer the user clearly and simply.\n"
        + context
        + "User: " + message.strip()
        + "\nAssistant:"
    )

    inputs = tokenizer(
        prompt, return_tensors="pt",
        truncation=True, max_length=512
    ).to(device)

    with torch.no_grad():
        result = model.generate(
            **inputs,
            max_new_tokens=100,
            num_beams=4,
            early_stopping=True
        )

    answer = tokenizer.decode(result[0], skip_special_tokens=True)
    return answer.strip() or "No response generated. Please try again."

if "messages" not in st.session_state:
    st.session_state.messages = []

with st.sidebar:
    st.header("Options")
    st.write("Model: FLAN-T5-small")
    if st.button("Clear Chat"):
        st.session_state.messages = []
        st.rerun()

    st.divider()
    st.subheader("Test Prompts")

    if st.button("Run 5 Test Prompts"):
        test_prompts = [
            "What is artificial intelligence?",
            "Explain machine learning simply.",
            "What is a chatbot?",
            "Give two benefits of cloud computing.",
            "Write three steps to learn Python."
        ]

        st.session_state.test_results = []
        for question in test_prompts:
            try:
                answer = generate_response(question)
                st.session_state.test_results.append((question, answer))
            except Exception as e:
                st.session_state.test_results.append(
                    (question, "Error: " + str(e))
                )

    if st.button("Clear Test Results"):
        st.session_state.test_results = []

# Display conversation history
for msg in st.session_state.messages:
    with st.chat_message(msg["role"]):
        st.write(msg["content"])

# Accept user prompt
user_input = st.chat_input("Enter your message...")

if user_input:
    st.session_state.messages.append({
        "role": "user", "content": user_input
    })

    with st.chat_message("user"):
        st.write(user_input)

    with st.chat_message("assistant"):
        try:
            with st.spinner("Generating response..."):
                history = st.session_state.messages[:-1]
                response = generate_response(user_input, history)
            st.write(response)
            st.session_state.messages.append({
                "role": "assistant", "content": response
            })
        except Exception as e:
            st.error("Unable to generate a response: " + str(e))

# Show five-prompt test output
if st.session_state.get("test_results"):
    st.divider()
    st.subheader("Test Results")
    for i, (question, answer) in enumerate(
        st.session_state.test_results, start=1
    ):
        st.markdown(f"**Test {i}: {question}**")
        st.write(answer)

st.divider()
st.caption("Note: LLM responses may be inaccurate. Verify important information.")
'''

with open("/content/app.py", "w", encoding="utf-8") as f:
    f.write(app_code)

# Stop previous Streamlit instance
subprocess.run(
    "pkill -f '[s]treamlit run /content/app.py'",
    shell=True,
    stdout=subprocess.DEVNULL,
    stderr=subprocess.DEVNULL
)
time.sleep(2)

# Start Streamlit
log = open("/content/streamlit.log", "w")
process = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "/content/app.py",
        "--server.port=8501",
        "--server.address=0.0.0.0",
        "--server.headless=true",
        "--server.enableCORS=false",
        "--server.enableXsrfProtection=false"
    ],
    stdout=log,
    stderr=subprocess.STDOUT
)

# Display app inside Colab
for attempt in range(40):
    try:
        with urllib.request.urlopen(
            "http://127.0.0.1:8501/_stcore/health", timeout=2
        ) as response:
            if response.status == 200:
                print("Chatbot started successfully!")
                output.serve_kernel_port_as_iframe(8501, height=750)
                break
    except Exception:
        time.sleep(1)
else:
    print("Could not start Streamlit. Check the logs:")
    print(open("/content/streamlit.log").read()[-3000:])


Chatbot started successfully!


<IPython.core.display.Javascript object>